# FANNIE MAE IFRS 9 COMPLIANCE FRAMEWORK
## 01: DATA LOADING & CLEANING

In Notebook 01, I built the foundation for the entire project. I took raw Fannie Mae data and transformed it into a clean, analysis-ready dataset.

1. **I started by setting up an audit trail — every decision is logged with a timestamp. Then I configured the key parameters: a 100,000-row sample with a fixed random seed for reproducibility.**

2. **I documented the CECL vs IFRS 9 context because Fannie Mae uses CECL, but I'm applying IFRS 9. I also documented the US/Canada applicability gap because the hiring manager might be Canadian.**

3. **I handled missing data with median imputation for numeric columns and mode for categorical — both robust and defensible choices. I handled outliers with winsorization at the 1st and 99th percentiles.**

4. **I created PD and LGD estimates that will be used for ECL calculation, and I created missing indicators for every column with missing data — that's an OSFI E-23 requirement.**

5. **Finally, I created a central assumptions register that documents every key decision, its rationale, and its regulatory reference. This is the single source of truth for the entire project.**

In [ ]:
"""
===============================================================================
FANNIE MAE IFRS 9 COMPLIANCE FRAMEWORK
NOTEBOOK 01: DATA LOADING & CLEANING
===============================================================================

AUTHOR: My Name is Milt
DATE: 2026-Sept-07
VERSION: 2.0

PURPOSE:
--------
This notebook loads raw Fannie Mae mortgage data, performs data quality
assessment, cleans missing values, handles outliers, and creates PD-LGD
estimates for use in the IFRS 9 ECL framework. It serves as the foundation
for all subsequent notebooks.

INPUTS:
-------
- Fannie Mae.zip (ZIP file containing mortgage performance data)
  - Expected CSV: train4.csv (or alternative CSV file)
- Configuration parameters defined in Section 1

OUTPUTS:
--------
- data/fannie_mae_raw_sample.csv         : Raw sampled data
- data/fannie_mae_cleaned.csv            : Cleaned data (missing values handled)
- data/fannie_mae_feature_engineered.csv : Feature-engineered data
- data/fannie_mae_final_clean.csv        : Final dataset ready for modeling
- data/assumptions_register.csv          : Central assumptions register
- outputs/data_quality_report_enhanced.txt
- outputs/data_cleaning_documentation.txt
- outputs/feature_engineering_documentation.txt
- outputs/target_variable_documentation.txt
- outputs/data_quality_dashboard.png
- outputs/default_pattern_analysis.png
- outputs/feature_engineering_results.png
- logs/ecl_audit_trail_{RUN_ID}.log

REGULATORY CONTEXT:
-------------------
- OSFI E-23 s.4.3: Data quality and integrity management
- OSFI E-23 s.4.2: Documentation of model development
- OSFI B-13 s.3.2: Reproducibility and audit trail
- IFRS 9: Data quality for ECL calculation

DATASET CAVEAT:
---------------
This project uses publicly available Fannie Mae data as a PROXY for
proprietary bank data. While the risk drivers (FICO, LTV, DTI, property type,
loan age) are representative of those used in bank portfolios, this is not
actual bank portfolio data. All conclusions are illustrative and demonstrate
methodology, not actual portfolio performance.

US/CANADA APPLICABILITY GAP:
----------------------------
Fannie Mae is a US GSE (Government-Sponsored Enterprise). For Canadian
application, see the US/Canada Applicability Gap section below.

"""

### Data Loading

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import zipfile
import os
import warnings
import logging
from datetime import datetime
import json
import matplotlib.pyplot as plt
import sys

warnings.filterwarnings('ignore')

# Set display options for better readability
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 30)

### AUDIT TRAIL - Setup logging module)

In [3]:
def setup_audit_logger(run_id=None):
    """Set up comprehensive audit trail logging."""
    if run_id is None:
        run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    os.makedirs('logs', exist_ok=True)
    
    logger = logging.getLogger(f"ecl_engine_{run_id}")
    logger.setLevel(logging.INFO)
    
    if logger.hasHandlers():
        logger.handlers.clear()
    
    handler = logging.FileHandler(f"logs/ecl_audit_trail_{run_id}.log", encoding='utf-8')
    formatter = logging.Formatter(
        '%(asctime)s | %(levelname)s | %(message)s', 
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)
    
    return logger, run_id

audit_logger, RUN_ID = setup_audit_logger()
audit_logger.info("="*60)
audit_logger.info("NOTEBOOK 01: DATA LOADING & CLEANING")
audit_logger.info(f"RUN ID: {RUN_ID}")
audit_logger.info("="*60)

print("\n✅ Audit trail initialized: logs/ecl_audit_trail_{RUN_ID}.log".format(RUN_ID=RUN_ID))

2026-09-07 10:35:34 | INFO | ============================================================
2026-09-07 10:35:34 | INFO | NOTEBOOK 01: DATA LOADING & CLEANING
2026-09-07 10:35:34 | INFO | RUN ID: 20260907_103534
2026-09-07 10:35:34 | INFO | ============================================================

✅ Audit trail initialized: logs/ecl_audit_trail_20260907_103534.log


**I understand that regulatory work requires an audit trail. I build traceability into my code from the start, not as an afterthought.**

#### Why This Decision Was Made:
- **Why a log file?** - OSFI B-13 requires an audit trail — every decision and action must be traceable. If a regulator asks "how did you choose the sample size?", the log shows the decision and timestamp.
- **Why the RUN_ID?** - Each run gets a unique ID (like 20260905_123014). This ensures logs from different runs don't overwrite each other. If you need to reproduce results, you can find the exact log for that run.
- **Why both console and file logging?** - Console logging shows you what's happening in real-time (good for debugging). File logging creates a permanent record for auditors.
- **Why the timestamp format?** - %Y-%m-%d %H:%M:%S creates a clear, sortable timestamp. An auditor can see exactly when each action occurred.

## SECTION 1: CONFIGURATION & CENTRAL ASSUMPTIONS REGISTER

### CONFIGURATION - Documenting Key Decisions

In [4]:
print("\n" + "="*60)
print("SECTION 1: CONFIGURATION & CENTRAL ASSUMPTIONS REGISTER")
print("="*60)

audit_logger.info("SECTION 1: CONFIGURATION")


SECTION 1: CONFIGURATION & CENTRAL ASSUMPTIONS REGISTER
2026-09-07 10:37:41 | INFO | SECTION 1: CONFIGURATION


### 1.1 DATA LOADING PARAMETERS

In [5]:
"""
DECISION: Sample Size
RATIONALE: 100,000 rows balances computational efficiency with statistical validity.
- Sufficient to capture default patterns (even with 1-2% default rate)
- Fast enough for rapid iteration during development
- Will use full dataset for production validation
ALTERNATIVE REJECTED: 10,000 rows (insufficient statistical power), 
                     1,000,000 rows (computationally prohibitive)
REGULATORY REFERENCE: OSFI B-13 s.3.2 (Reproducibility)
"""
SAMPLE_SIZE = 100000
audit_logger.info(f"Sample Size: {SAMPLE_SIZE:,} rows")


"""
DECISION: Random Seed
RATIONALE: Fixed seed ensures reproducibility and consistent results.
- Stakeholders can reproduce our analysis
- Allows comparison between different runs
- Standard practice in model risk governance
ALTERNATIVE REJECTED: No seed (non-reproducible), different seed values (would change results)
REGULATORY REFERENCE: OSFI B-13 s.3.2 (Reproducibility)
"""
RANDOM_STATE = 42
audit_logger.info(f"Random Seed: {RANDOM_STATE}")


"""
DECISION: File Configuration
RATIONALE: Known file structure from Fannie Mae data documentation
- ZIP file contains the raw data
- CSV file is the primary data source
REGULATORY REFERENCE: OSFI E-23 s.4.2 (Documentation)
"""
ZIP_FILE = 'Fannie Mae.zip'
CSV_FILE = 'train4.csv'

# Create necessary directories
os.makedirs('outputs', exist_ok=True)
os.makedirs('data', exist_ok=True)

print("\nConfiguration Summary:")
print(f"  Sample Size: {SAMPLE_SIZE:,} rows")
print(f"  Random Seed: {RANDOM_STATE}")
print(f"  Data Source: {ZIP_FILE} -> {CSV_FILE}")
print(f"  Run ID: {RUN_ID}")

audit_logger.info(f"Configuration Summary: Sample Size={SAMPLE_SIZE}, Random Seed={RANDOM_STATE}")

2026-09-07 10:37:44 | INFO | Sample Size: 100,000 rows
2026-09-07 10:37:44 | INFO | Random Seed: 42

Configuration Summary:
  Sample Size: 100,000 rows
  Random Seed: 42
  Data Source: Fannie Mae.zip -> train4.csv
  Run ID: 20260907_103534
2026-09-07 10:37:44 | INFO | Configuration Summary: Sample Size=100000, Random Seed=42


#### What It Means

This sets the parameters for the entire project:

- 100,000 rows of data will be sampled
- Random seed 42 ensures the same random sample every time
- Fannie Mae.zip is the source data file

#### Why These Decisions Were Made
                        
- **Why 100,000 rows?** - The full dataset has 1.67 million rows. 100,000 is large enough to capture default patterns (even with a 0.94% default rate) but small enough to run quickly during development. In production, you'd use the full dataset.
- **Why random seed 42?** - "42" is a common placeholder (from Hitchhiker's Guide). The actual number doesn't matter — what matters is that it's fixed. This ensures reproducibility.
- **Why random sampling instead of sequential?** - The data is ordered by origination date. If you took the first 100,000 rows, you'd only have loans from the earliest years. Random sampling ensures representation across all time periods.
- **Why Fannie Mae.zip?** - This is the known file structure from Fannie Mae's public data. It's documented in their data dictionary.

**I think about reproducibility before I write a single line of code. I document my decisions with rationale.**

### 1.2 CENTRAL ASSUMPTIONS REGISTER - MASTER TABLE

In [6]:
"""
===============================================================================
CENTRAL ASSUMPTIONS REGISTER
===============================================================================
This register documents every threshold, parameter, and assumption used across
all notebooks. It serves as the single source of truth for audit and validation
purposes.

Regulatory Reference: OSFI E-23 s.4.2 (Documentation), OSFI B-13 s.3.2 (Audit Trail)

===============================================================================
DATA PREPARATION PARAMETERS
===============================================================================
| Parameter                  | Value        | Rationale                                      | Regulatory Reference     | Notebook Cell     |
|----------------------------|--------------|------------------------------------------------|--------------------------|-------------------|
| RANDOM_STATE               | 42           | Ensures reproducibility                        | OSFI B-13 s.3.2          | Section 1.1       |
| SAMPLE_SIZE                | 100,000      | Balances efficiency vs. statistical validity   | OSFI E-23 s.4.3          | Section 1.1       |
| Missing Data Threshold     | >50%         | Drop columns with >50% missing                 | OSFI E-23 s.4.3          | Section 3.3       |
| Numeric Imputation Method  | Median       | Robust to outliers, preserves distribution     | OSFI E-23 s.4.3          | Section 3.3       |
| Categorical Imputation     | Mode         | Preserves category distribution                | OSFI E-23 s.4.3          | Section 3.3       |
| Outlier Treatment          | Winsorization| 1st/99th percentiles, less destructive         | OSFI E-23 s.4.3          | Section 3.3       |
| LTV Threshold              | 80%          | Aligns with mortgage insurance (PMI)           | HMDA, PMI requirements   | Section 4.1       |
| FICO Cutoffs               | 680,720,760  | Industry standard for risk segmentation        | Fannie Mae risk bands    | Section 4.1       |
| Default Definition         | 90+ DPD      | Industry standard for mortgage default         | FFIEC, OCC, IFRS 9       | Section 4.1       |
===============================================================================
"""


def create_assumptions_register():
    """Create the central assumptions register as a DataFrame."""
    assumptions_data = [
        # [Parameter, Value, Rationale, Regulatory Reference, Notebook, Section]
        ["RANDOM_STATE", "42", "Ensures reproducibility across all stochastic operations", "OSFI B-13 s.3.2", "01", "Section 1.1"],
        ["SAMPLE_SIZE", "100,000", "Balances computational efficiency with statistical validity", "OSFI E-23 s.4.3", "01", "Section 1.1"],
        ["Missing Data Threshold", ">50%", "Drop columns with >50% missing values", "OSFI E-23 s.4.3", "01", "Section 3.3"],
        ["Numeric Imputation", "Median", "Robust to outliers, preserves distribution", "OSFI E-23 s.4.3", "01", "Section 3.3"],
        ["Categorical Imputation", "Mode", "Preserves category distribution", "OSFI E-23 s.4.3", "01", "Section 3.3"],
        ["Outlier Treatment", "Winsorization (1st/99th)", "Less destructive than removal, common in risk modeling", "OSFI E-23 s.4.3", "01", "Section 3.3"],
        ["LTV Threshold", "80%", "Aligns with mortgage insurance (PMI) requirement", "HMDA, PMI", "01", "Section 4.1"],
        ["FICO Cutoffs", "680, 720, 760", "Industry standard for mortgage risk segmentation", "Fannie Mae risk bands", "01", "Section 4.1"],
        ["Default Definition", "90+ DPD", "Industry standard for mortgage default", "FFIEC, OCC, IFRS 9", "01", "Section 4.1"],
        ["CECL vs IFRS 9", "IFRS 9 applied to CECL-source data", "Demonstrate IFRS 9 methodology using representative data", "IFRS 9 s.5.5", "01", "Section 2.1"],
    ]
    
    assumptions_df = pd.DataFrame(assumptions_data, columns=[
        'Parameter', 'Value', 'Rationale', 'Regulatory_Reference', 'Notebook', 'Section'
    ])
    
    return assumptions_df


# Create and save the assumptions register
assumptions_df = create_assumptions_register()
assumptions_df.to_csv('data/assumptions_register.csv', index=False)
print("\n✅ Created and saved: data/assumptions_register.csv")
audit_logger.info("Created assumptions register with {} entries".format(len(assumptions_df)))

# Display the register
print("\nCentral Assumptions Register:")
print(assumptions_df.to_string(index=False))


✅ Created and saved: data/assumptions_register.csv
2026-09-07 10:40:04 | INFO | Created assumptions register with 10 entries

Central Assumptions Register:
             Parameter                              Value                                                   Rationale  Regulatory_Reference Notebook     Section
          RANDOM_STATE                                 42    Ensures reproducibility across all stochastic operations       OSFI B-13 s.3.2       01 Section 1.1
           SAMPLE_SIZE                            100,000 Balances computational efficiency with statistical validity       OSFI E-23 s.4.3       01 Section 1.1
Missing Data Threshold                               >50%                       Drop columns with >50% missing values       OSFI E-23 s.4.3       01 Section 3.3
    Numeric Imputation                             Median                  Robust to outliers, preserves distribution       OSFI E-23 s.4.3       01 Section 3.3
Categorical Imputation                

#### What It Means

This creates a central register of every key assumption in your project. It's a single source of truth that answers: "What did we assume, why, and what regulation does it satisfy?"

#### Why This Decision Was Made
- **Why a central register?** - In a real bank, model documentation is scattered across multiple documents. A central register ensures traceability — you can find every assumption in one place.
- **Why include regulatory reference?** - This shows regulatory awareness — you're not just making arbitrary choices; you're aligning with specific regulations.
- **Why include notebook and section?** - If someone needs to find the code where the assumption is implemented, they can go directly to the right notebook and section.
- **Why save as CSV?** - CSV is a universal format that can be opened in Excel, loaded into Power BI, or read by any programming language. It's accessible to non-technical stakeholders.

**I understand that model risk management requires documentation. I create a single source of truth that anyone can access and understand.**

### SECTION 2: CECL vs IFRS 9 CONTEXT

In [7]:
print("\n" + "="*60)
print("SECTION 2: CECL vs IFRS 9 CONTEXT")
print("="*60)

audit_logger.info("SECTION 2: CECL vs IFRS 9 CONTEXT")

print("""
===============================================================================
IMPORTANT: CECL vs IFRS 9 — Project Context and Data Mapping
===============================================================================

WHAT THIS PROJECT IS:
This project applies IFRS 9 methodology to publicly available Fannie Mae
mortgage data as a PROXY for a bank portfolio.

===============================================================================
THE KEY DIFFERENCE: CECL vs IFRS 9
===============================================================================

| Aspect                  | CECL (Fannie Mae)              | IFRS 9 (This Project)              |
|-------------------------|--------------------------------|------------------------------------|
| Approach                | Lifetime expected loss from    | Staged approach based on credit    |
|                         | origination                    | deterioration                      |
| Stage 1                 | N/A (all loans are lifetime)   | 12-month ECL for performing loans  |
| Stage 2                 | N/A                            | Lifetime ECL for SICR loans        |
| Stage 3                 | N/A                            | Lifetime ECL for credit-impaired   |
| PD Definition           | Lifetime PD from origination   | 12-month PD (Stage 1) vs Lifetime  |
|                         |                                | PD (Stages 2/3)                    |
| Data Requirement        | Origination data + lifetime    | Origination data + performance     |
|                         | performance                    | data to identify SICR              |

===============================================================================
WHY THIS MAPPING IS VALID
===============================================================================

Although Fannie Mae uses CECL, the underlying data and risk drivers are the
same type used by banks for IFRS 9:

- Origination data: FICO, LTV, DTI, property type, loan amount, term
- Performance data: Default status, delinquency state, age
- Risk drivers: Credit quality, collateral, borrower characteristics

The data contains all the fields needed to implement IFRS 9 staging:
- Loan age (AGE) allows us to assess seasoning and vintage effects
- Default status (default) allows us to calibrate PD
- Risk drivers (FICO, LTV, DTI) allow us to segment and calibrate PD/LGD/EAD
- Delinquency state (delinquency_state) allows us to classify Stage 2/3

===============================================================================
WHAT THIS PROJECT DOES NOT CLAIM
===============================================================================

1. This project does NOT claim to be a production IFRS 9 implementation
   for a real bank

2. This project does NOT claim that Fannie Mae uses IFRS 9

3. This project does NOT claim that the PD/LGD/EAD estimates are actual
   bank portfolio estimates

4. This project does NOT claim to be representative of any specific
   bank portfolio

===============================================================================
WHAT THIS PROJECT DOES CLAIM
===============================================================================

1. This project demonstrates the METHODOLOGY of IFRS 9 ECL calculation

2. This project uses REAL-WORLD RISK DRIVERS that are representative of
   bank portfolios

3. This project shows how to IMPLEMENT IFRS 9 CONCEPTS using mortgage data

4. This project is a TRAINING/PORTFOLIO EXERCISE demonstrating regulatory
   compliance capabilities

===============================================================================
""")


SECTION 2: CECL vs IFRS 9 CONTEXT
2026-09-07 10:42:29 | INFO | SECTION 2: CECL vs IFRS 9 CONTEXT

IMPORTANT: CECL vs IFRS 9 — Project Context and Data Mapping

WHAT THIS PROJECT IS:
This project applies IFRS 9 methodology to publicly available Fannie Mae
mortgage data as a PROXY for a bank portfolio.

THE KEY DIFFERENCE: CECL vs IFRS 9

| Aspect                  | CECL (Fannie Mae)              | IFRS 9 (This Project)              |
|-------------------------|--------------------------------|------------------------------------|
| Approach                | Lifetime expected loss from    | Staged approach based on credit    |
|                         | origination                    | deterioration                      |
| Stage 1                 | N/A (all loans are lifetime)   | 12-month ECL for performing loans  |
| Stage 2                 | N/A                            | Lifetime ECL for SICR loans        |
| Stage 3                 | N/A                            | Lifetime EC

#### What It Means

This is a disclaimer and honesty section. It explicitly states:

- Fannie Mae uses CECL (not IFRS 9)
- You are applying IFRS 9 to CECL-source data
- Why this is valid (the data contains the same risk drivers)

What you are NOT claiming (not a production implementation, not actual bank data)

#### Why This Decision Was Made

- **Why is this the most important section?** - This is the single most important honesty requirement for your project. Without this, a hiring manager or regulator might assume you're presenting actual IFRS 9 results from a real bank. That would be misleading.
- **Why the comparison table?** - A table makes the difference immediately clear to any reader. It's professional and concise.
- **Why state what you're NOT claiming?** - This preempts misconceptions. It's better to be explicit about limitations than to let someone assume incorrectly.
- **Why state what you ARE claiming?** - This reframes your project — you're demonstrating methodology, not delivering a production implementation. That's a valid and valuable contribution.

**I am honest and self-aware. I understand the limitations of my work and communicate them clearly. I don't want to make overclaim.**

### SECTION 2.1: US/CANADA APPLICABILITY GAP

In [8]:
print("\n" + "="*60)
print("SECTION 2.1: US/CANADA APPLICABILITY GAP")
print("="*60)

audit_logger.info("SECTION 2.1: US/CANADA APPLICABILITY GAP")

print("""
===============================================================================
US/CANADA APPLICABILITY GAP
===============================================================================

Important Context for Canadian Readers:
---------------------------------------
This project uses US Fannie Mae data to demonstrate IFRS 9 methodology.
For a Canadian audience, there are important differences to note.

===============================================================================
KEY DIFFERENCES: US vs CANADIAN MORTGAGE MARKETS
===============================================================================

| Aspect                     | US (Fannie Mae Data)          | Canada                          | Impact on Model                    |
|----------------------------|-------------------------------|---------------------------------|------------------------------------|
| Government-Sponsored       | Fannie Mae and Freddie Mac    | No Canadian GSE                 | Data source not representative of  |
| Enterprise (GSE)           | (public data available)       | (CMHC is a Crown corporation)   | Canadian market                    |
| Mortgage Insurance         | PMI for >80% LTV              | CMHC insurance for >80% LTV    | Loss severities differ             |
|                            | (private insurance)           | (government-backed)             | (CMHC-backed loans have lower LGD) |
| Stress Test                | No national stress test       | B-20 stress test (qualifying    | Default behavior differs under     |
|                            |                               | rate 200bps above contract)    | stress                             |
| Typical Mortgage Term     | 30-year fixed rate            | 5-year term, 25-year            | PD/LGD dynamics differ             |
|                            | most common                   | amortization most common        | (more frequent refinancing)        |
| Portability                | Mortgages are not portable    | Mortgages are portable          | Default risk differs               |
|                            |                               | (can transfer to new property)  | (less incentive to default)        |
| Recourse                   | Non-recourse in some states   | Full recourse in all provinces  | LGD differs                        |
|                            |                               |                                 | (higher recovery in Canada)        |
| Data Availability          | Fannie Mae public data        | No equivalent public mortgage   | Reproducibility challenge          |
|                            | available                     | performance data                |                                    |
| Regulatory Environment     | OCC, Federal Reserve, CFPB    | OSFI, CMHC, FCAC                | Regulatory references must be      |
|                            |                               |                                 | adapted for Canadian context       |

===============================================================================
IMPLICATIONS FOR MODEL CALIBRATION
===============================================================================

The following model parameters would need to be recalibrated for a Canadian
portfolio:

| Parameter      | US (Fannie Mae) Calibration      | Canadian Adjustment Required           |
|----------------|----------------------------------|----------------------------------------|
| PD             | Based on Fannie Mae historical   | Use Canadian bank data (e.g., OSFI     |
|                | default rates                    | mortgage data)                         |
| LGD            | Based on US recovery rates       | CMHC insurance reduces LGD for >80%    |
|                | (varies by state)                | LTV loans                              |
| EAD            | Based on US prepayment behavior  | Canadian prepayment behavior differs   |
|                |                                  | (5-year term)                          |
| LTV Thresholds | 80% (PMI requirement)            | 80% (CMHC requirement) — similar       |
| Stress Testing | Based on US economic scenarios   | Use OSFI stress scenarios              |

===============================================================================
WHAT THIS PROJECT DOES NOT CLAIM (CANADIAN CONTEXT)
===============================================================================

1. This project does NOT claim to be representative of the Canadian
   mortgage market

2. This project does NOT claim that Fannie Mae data is a substitute for
   Canadian data

3. This project does NOT claim that OSFI E-23 is a US regulation

4. This project does NOT claim that the PD/LGD/EAD estimates are applicable
   to Canadian portfolios

5. This project does NOT claim that Canadian regulatory thresholds
   (e.g., OSFI B-20 stress test) have been applied

===============================================================================
WHAT THIS PROJECT DOES DEMONSTRATE (CANADIAN CONTEXT)
===============================================================================

1. This project demonstrates the METHODOLOGY of IFRS 9 ECL calculation
   using mortgage data

2. This project shows how to IMPLEMENT OSFI E-23 documentation standards

3. This project illustrates REGULATORY COMPLIANCE CAPABILITIES applicable
   to both US and Canadian contexts

4. This project demonstrates RISK DRIVER IDENTIFICATION (FICO, LTV, DTI, age)
   that is relevant to any mortgage portfolio

5. This project shows CANADIAN MARKET AWARENESS by explicitly identifying
   the US/Canada differences

===============================================================================
RECOMMENDED NEXT STEPS FOR CANADIAN ADAPTATION
===============================================================================

1. Replace data source: Use Canadian bank data or OSFI's publicly available
   mortgage data

2. Recalibrate PD: Use OSFI historical default rates for Canadian portfolios

3. Recalibrate LGD: Consider CMHC insurance and full recourse environment

4. Apply B-20 stress test: Incorporate the B-20 qualifying rate stress test

5. Use OSFI scenarios: Replace US economic scenarios with OSFI-defined
   scenarios

6. Validate with Canadian historical data: Compare results to actual
   Canadian mortgage performance

===============================================================================

CAVEAT: This project is a DEMONSTRATION OF METHODOLOGY, not a production
implementation for either the US or Canadian market. The US/Canada gap is
acknowledged here to provide honest context for Canadian readers.

===============================================================================
""")


SECTION 2.1: US/CANADA APPLICABILITY GAP
2026-09-07 10:46:10 | INFO | SECTION 2.1: US/CANADA APPLICABILITY GAP

US/CANADA APPLICABILITY GAP

Important Context for Canadian Readers:
---------------------------------------
This project uses US Fannie Mae data to demonstrate IFRS 9 methodology.
For a Canadian audience, there are important differences to note.

KEY DIFFERENCES: US vs CANADIAN MORTGAGE MARKETS

| Aspect                     | US (Fannie Mae Data)          | Canada                          | Impact on Model                    |
|----------------------------|-------------------------------|---------------------------------|------------------------------------|
| Government-Sponsored       | Fannie Mae and Freddie Mac    | No Canadian GSE                 | Data source not representative of  |
| Enterprise (GSE)           | (public data available)       | (CMHC is a Crown corporation)   | Canadian market                    |
| Mortgage Insurance         | PMI for >80% LTV      

#### What It Means

This acknowledges that Canada is different from the US for mortgage markets. It's a "Canadian market awareness" section that shows you understand the differences:

- No Canadian GSE (Fannie Mae equivalent)
- CMHC insurance (vs US PMI)
- B-20 stress test (vs no US national stress test)
- 5-year term (vs 30-year fixed rate)
- Full recourse (vs non-recourse in some US states)

#### Why This Decision Was Made

- **Why is this important?** - If you're applying for a role in Canada, the hiring manager wants to know you understand the Canadian context. This section proves you do.
- **Why the comparison table?** - A table makes the differences immediately visible. It shows you've thought through the implications.
- **Why include "Impact on Model"?** - This shows you understand why these differences matter — they affect PD, LGD, EAD, and stress testing.
- **Why include "Recommended Next Steps"?** - This shows you're practical — you don't just identify problems; you propose solutions.

**I understand the Canadian market. I know about CMHC, B-20, and the lack of a Canadian GSE. I can adapt US methodology to Canadian context.**

## SECTION 3: DATA LOADING

#### Part 1: Synthetic Data Fallback

In [9]:
print("\n" + "="*60)
print("SECTION 3: DATA LOADING")
print("="*60)

audit_logger.info("SECTION 3: DATA LOADING")


def create_synthetic_fannie_mae_data(n_samples, random_state=42):
    """
    Create synthetic Fannie Mae-like data with PD-LGD estimates.
    
    DECISION: Use synthetic data when real data is unavailable
    RATIONALE: 
    - Ensures the project runs even without real data
    - Provides consistent results for portfolio demonstration
    - Mimics real data distributions to test the pipeline
    - Allows focus on methodology rather than data access
    
    Parameters:
    -----------
    n_samples : int
        Number of synthetic records to generate
    random_state : int
        Seed for reproducibility
    
    Returns:
    --------
    pd.DataFrame : Synthetic loan data with PD-LGD estimates
    """
    np.random.seed(random_state)
    
    print(f"\nCreating synthetic data with {n_samples:,} rows...")
    audit_logger.info(f"Creating synthetic data with {n_samples:,} rows")
    
    # Generate core loan characteristics
    data = {
        'ID': np.arange(1, n_samples + 1),
        'ORG_UPB': np.round(np.random.uniform(50000, 500000, n_samples), 2),
        'ORG_LTV': np.round(np.random.normal(70, 15, n_samples).clip(10, 97), 1),
        'DTI': np.round(np.random.normal(30, 10, n_samples).clip(2, 50), 1),
        'FICO_BOR': np.random.normal(770, 40, n_samples).clip(620, 832).astype(int),
        'AGE': np.random.uniform(6, 36, n_samples),
        'PROP_TYPE': np.random.choice(['SF', 'PU', 'CO', 'MH', 'CP'], 
                                      n_samples, p=[0.7, 0.15, 0.1, 0.04, 0.01]),
        'OCCU_STAT': np.random.choice(['P', 'S', 'I'], n_samples, p=[0.7, 0.2, 0.1]),
        'CUR_UBP': np.round(np.random.uniform(50000, 500000, n_samples), 2),
        'ORG_TERM': np.random.choice([180, 240, 300, 360], n_samples, 
                                     p=[0.1, 0.15, 0.25, 0.5]),
    }
    
    df = pd.DataFrame(data)
    
    # Create default indicator based on risk factors
    risk_score = (
        (1 - (df['FICO_BOR'] - 620) / 212) * 0.4 +
        (df['ORG_LTV'] / 100) * 0.3 +
        (df['DTI'] / 50) * 0.2 +
        (df['AGE'] / 36) * 0.1
    )
    default_prob = 0.01 + 0.09 * risk_score
    df['default'] = (np.random.random(n_samples) < default_prob).astype(int)
    
    # Create PD estimate
    df['pd_estimate'] = 0.01 + 0.09 * (risk_score / max(risk_score) if max(risk_score) > 0 else 1)
    df['pd_estimate'] = df['pd_estimate'].clip(0.01, 0.20)
    
    # Create LGD estimate with PD correlation
    def lgd_adjustment_by_pd(pd_estimate, min_lgd=0.10, max_lgd=0.90):
        pd_scaled = np.clip(pd_estimate, 0, 1)
        return min_lgd + (max_lgd - min_lgd) * pd_scaled
    
    df['lgd_correlated'] = lgd_adjustment_by_pd(df['pd_estimate'])
    
    # Property type adjustments
    property_lgd_adjustments = {
        'SF': -0.05, 'PU': 0.00, 'CO': 0.10, 'MH': 0.20, 'CP': 0.15
    }
    df['lgd_final'] = df['lgd_correlated'] + df['PROP_TYPE'].map(property_lgd_adjustments).fillna(0)
    df['lgd_final'] = df['lgd_final'].clip(0.10, 0.90)
    
    # Multi-state delinquency
    df['delinquency_state'] = df['default'].astype(int) * 5
    state_labels = {0: 'Current', 5: '180+ DPD'}
    df['delinquency_label'] = df['delinquency_state'].map(state_labels).fillna('Current')
    
    # Ratio features
    df['upb_ratio'] = df['CUR_UBP'] / df['ORG_UPB']
    
    # Non-linear risk features
    df['risk_fico'] = 1 / (1 + np.exp(0.05 * (df['FICO_BOR'] - 720)))
    df['risk_ltv'] = 1 / (1 + np.exp(-0.15 * (df['ORG_LTV'] - 80)))
    
    print(f"  Generated {len(df):,} rows with {len(df.columns)} columns")
    print(f"  Default Rate: {df['default'].mean():.2%}")
    print(f"  PD-LGD Correlation: {df['pd_estimate'].corr(df['lgd_final']):.3f}")
    
    audit_logger.info(f"Generated {len(df):,} rows with {len(df.columns)} columns")
    audit_logger.info(f"Default Rate: {df['default'].mean():.2%}")
    audit_logger.info(f"PD-LGD Correlation: {df['pd_estimate'].corr(df['lgd_final']):.3f}")
    
    return df



SECTION 3: DATA LOADING
2026-09-07 10:48:22 | INFO | SECTION 3: DATA LOADING


#### What It Means

If the real Fannie Mae data file is not found, this function creates fake but realistic-looking mortgage data. It's a fallback to ensure the project runs.

#### Why This Decision Was Made
- **Why synthetic data?** - The project should run even if someone doesn't have the Fannie Mae ZIP file. This makes the project self-contained and portable.
- **Why mimic real data?** - The synthetic data is generated with realistic distributions (FICO 620-832, LTV 10-97, DTI 2-50) so the methodology is still demonstrated correctly.
- **Why the random seed?** - Even synthetic data should be reproducible. The same seed produces the same synthetic data every time.
- **Why PD-LGD correlation?** - Real data has a positive correlation between PD and LGD (higher default risk = higher loss severity). The synthetic data replicates this.

**I think about edge cases and fallbacks. Even if data isn't available, the project still works and demonstrates the methodology.**

### Part 2: Real Data Loading

In [10]:
def load_fannie_mae_data(zip_path, csv_file, sample_size=100000, random_state=42):
    """
    Load Fannie Mae data and create PD-LGD estimates.
    
    DECISION: Use random sampling instead of sequential loading
    RATIONALE:
    - Fannie Mae data is chronologically ordered by origination date
    - Sequential loading would only include early originations
    - Random sampling ensures representation across all time periods
    - Prevents temporal bias in model development
    
    Parameters:
    -----------
    zip_path : str
        Path to the ZIP file
    csv_file : str
        Name of CSV file inside ZIP
    sample_size : int
        Number of rows to sample
    random_state : int
        Seed for reproducibility
    
    Returns:
    --------
    tuple : (DataFrame, data_source_string)
    """
    print(f"\nAttempting to load data from: {zip_path}")
    audit_logger.info(f"Attempting to load data from: {zip_path}")
    
    if not Path(zip_path).exists():
        print(f"  ZIP file not found: {zip_path}")
        print("  Falling back to synthetic data...")
        audit_logger.warning(f"ZIP file not found: {zip_path} - using synthetic data")
        return create_synthetic_fannie_mae_data(sample_size), "Synthetic Data"
    
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            print(f"  Found ZIP file: {zip_path}")
            audit_logger.info(f"Found ZIP file: {zip_path}")
            
            if csv_file in z.namelist():
                print(f"  Using CSV: {csv_file}")
                target_file = csv_file
            else:
                csv_files = [f for f in z.namelist() if f.endswith('.csv')]
                if csv_files:
                    print(f"  Found alternative CSV: {csv_files[0]}")
                    target_file = csv_files[0]
                else:
                    print("  No CSV files found in ZIP")
                    print("  Falling back to synthetic data...")
                    audit_logger.warning("No CSV files found in ZIP - using synthetic data")
                    return create_synthetic_fannie_mae_data(sample_size), "Synthetic Data"
            
            print(f"\n  Reading data...")
            audit_logger.info(f"Reading CSV: {target_file}")
            
            with z.open(target_file) as f:
                df = pd.read_csv(f, low_memory=False)
            
            total_rows = len(df)
            print(f"  Total rows available: {total_rows:,}")
            audit_logger.info(f"Total rows available: {total_rows:,}")
            
            print(f"  Taking random sample of {sample_size:,} rows...")
            df_sample = df.sample(n=min(sample_size, total_rows), random_state=random_state)
            
            print(f"\n  ✅ Data loaded successfully!")
            print(f"  Source: Real Fannie Mae Data (Random Sample)")
            print(f"  Shape: {df_sample.shape}")
            
            audit_logger.info(f"Data loaded: {df_sample.shape}")
            
            # ================================================================
            # CREATE/ENSURE TARGET VARIABLE
            # ================================================================
            
            target_found = False
            target_candidates = ['default', 'TARGET', 'Default', 'DEFAULT', 
                               'is_default', 'IsDefault', 'bad', 'BAD']
            
            for col in target_candidates:
                if col in df_sample.columns:
                    if col != 'default':
                        df_sample['default'] = df_sample[col]
                    print(f"  Found target variable: '{col}' -> renamed to 'default'")
                    audit_logger.info(f"Found target variable: '{col}' -> renamed to 'default'")
                    target_found = True
                    break
            
            if not target_found:
                if 'CUR_UBP' in df_sample.columns and 'ORG_UPB' in df_sample.columns:
                    print("  No target variable found. Creating from UPB ratio...")
                    upb_ratio = df_sample['CUR_UBP'] / df_sample['ORG_UPB']
                    df_sample['default'] = (upb_ratio < 0.7).astype(int)
                    print(f"  Default Rate: {df_sample['default'].mean():.2%}")
                    audit_logger.info("Created default from UPB ratio")
                    target_found = True
            
            if not target_found:
                print("  No target variable found. Creating synthetic default...")
                audit_logger.warning("No target found - creating synthetic default")
                risk_score = np.zeros(len(df_sample))
                if 'FICO_BOR' in df_sample.columns:
                    risk_score += (1 - (df_sample['FICO_BOR'] - 620) / 212) * 0.4
                if 'ORG_LTV' in df_sample.columns:
                    risk_score += (df_sample['ORG_LTV'] / 100) * 0.3
                if 'DTI' in df_sample.columns:
                    risk_score += (df_sample['DTI'] / 50) * 0.2
                if 'AGE' in df_sample.columns:
                    risk_score += (df_sample['AGE'] / 36) * 0.1
                
                default_prob = 0.01 + 0.09 * (risk_score / max(risk_score) if max(risk_score) > 0 else 1)
                df_sample['default'] = (np.random.random(len(df_sample)) < default_prob).astype(int)
                print(f"  Default Rate: {df_sample['default'].mean():.2%}")
            
            # ================================================================
            # CREATE PD-LGD ESTIMATES FOR REAL DATA
            # ================================================================
            
            print("\n  Creating PD-LGD estimates for real data...")
            audit_logger.info("Creating PD-LGD estimates for real data")
            
            # Create risk score
            risk_score = np.zeros(len(df_sample))
            
            if 'FICO_BOR' in df_sample.columns:
                risk_score += (1 - (df_sample['FICO_BOR'] - 620) / 212) * 0.4
            if 'ORG_LTV' in df_sample.columns:
                risk_score += (df_sample['ORG_LTV'] / 100) * 0.3
            if 'DTI' in df_sample.columns:
                risk_score += (df_sample['DTI'] / 50) * 0.2
            if 'AGE' in df_sample.columns:
                risk_score += (df_sample['AGE'] / 36) * 0.1
            
            # Normalize risk score
            risk_score = risk_score / max(risk_score) if max(risk_score) > 0 else risk_score
            
            # Create PD estimate (0.01% to 20%)
            df_sample['pd_estimate'] = 0.01 + 0.09 * risk_score
            df_sample['pd_estimate'] = df_sample['pd_estimate'].clip(0.01, 0.20)
            
            # Create LGD estimate with PD correlation
            def lgd_adjustment_by_pd(pd_estimate, min_lgd=0.10, max_lgd=0.90):
                pd_scaled = np.clip(pd_estimate, 0, 1)
                return min_lgd + (max_lgd - min_lgd) * pd_scaled
            
            df_sample['lgd_correlated'] = lgd_adjustment_by_pd(df_sample['pd_estimate'])
            
            # Property type adjustments (if available)
            property_lgd_adjustments = {
                'SF': -0.05, 'PU': 0.00, 'CO': 0.10, 'MH': 0.20, 'CP': 0.15
            }
            if 'PROP_TYPE' in df_sample.columns:
                df_sample['lgd_final'] = df_sample['lgd_correlated'] + df_sample['PROP_TYPE'].map(property_lgd_adjustments).fillna(0)
            else:
                df_sample['lgd_final'] = df_sample['lgd_correlated']
            
            df_sample['lgd_final'] = df_sample['lgd_final'].clip(0.10, 0.90)
            
            pd_lgd_corr = df_sample['pd_estimate'].corr(df_sample['lgd_final'])
            print(f"    PD-LGD Correlation: {pd_lgd_corr:.3f}")
            audit_logger.info(f"PD-LGD Correlation: {pd_lgd_corr:.3f}")
            
            # ================================================================
            # ADD ADDITIONAL ENHANCED FEATURES
            # ================================================================
            
            # Multi-state delinquency
            df_sample['delinquency_state'] = df_sample['default'].astype(int) * 5
            state_labels = {0: 'Current', 5: '180+ DPD'}
            df_sample['delinquency_label'] = df_sample['delinquency_state'].map(state_labels).fillna('Current')
            
            # Ratio features
            if 'CUR_UBP' in df_sample.columns and 'ORG_UPB' in df_sample.columns:
                df_sample['upb_ratio'] = df_sample['CUR_UBP'] / df_sample['ORG_UPB']
            
            # Non-linear risk features
            if 'FICO_BOR' in df_sample.columns:
                df_sample['risk_fico'] = 1 / (1 + np.exp(0.05 * (df_sample['FICO_BOR'] - 720)))
            if 'ORG_LTV' in df_sample.columns:
                df_sample['risk_ltv'] = 1 / (1 + np.exp(-0.15 * (df_sample['ORG_LTV'] - 80)))
            
            print(f"  ✅ Enhanced features created for real data")
            audit_logger.info("Enhanced features created for real data")
            
            return df_sample, "Real Fannie Mae Data (Random Sample)"
            
    except Exception as e:
        print(f"  ❌ Error loading data: {e}")
        audit_logger.error(f"Error loading data: {e} - using synthetic data")
        return create_synthetic_fannie_mae_data(sample_size), "Synthetic Data"

### What It Means

This loads the real Fannie Mae data from the ZIP file, takes a random sample, and creates PD-LGD estimates.

#### Why These Decisions Were Made
- **Why low_memory=False?** - The CSV has many columns and rows. low_memory=False forces pandas to read the entire file at once, preventing data type inference errors.
- **Why random sampling?** - As explained earlier — prevents temporal bias.
- **Why create PD-LGD estimates here?** - The raw Fannie Mae data doesn't have PD or LGD columns. You create them using risk factors (FICO, LTV, DTI) to make the data usable for IFRS 9 modeling.
- **Why the if 'CUR_UBP' in df.columns fallback?** - If the target variable (default) isn't found, you create it from the UPB ratio. This makes the pipeline robust to different data formats.

**I write robust code that handles different data formats gracefully. I think about edge cases and build fallbacks.**

### SECTION 4: EXECUTE DATA LOADING

In [11]:
print("\n" + "-"*60)
print("EXECUTING DATA LOADING")
print("-"*60)

audit_logger.info("Executing data loading")

df, DATA_SOURCE = load_fannie_mae_data(
    zip_path=ZIP_FILE,
    csv_file=CSV_FILE,
    sample_size=SAMPLE_SIZE,
    random_state=RANDOM_STATE
)

print("\n" + "="*60)
print("LOADING RESULTS")
print("="*60)
print(f"\nData Source: {DATA_SOURCE}")
print(f"Dataset Shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
print(f"Default Rate: {df['default'].mean():.2%}")

# Verify PD-LGD columns exist
if 'pd_estimate' in df.columns and 'lgd_final' in df.columns:
    pd_lgd_corr = df['pd_estimate'].corr(df['lgd_final'])
    print(f"PD-LGD Correlation: {pd_lgd_corr:.3f}")
    audit_logger.info(f"PD-LGD Correlation verified: {pd_lgd_corr:.3f}")
else:
    print("⚠️  WARNING: PD-LGD columns are missing!")
    audit_logger.warning("PD-LGD columns are missing!")

audit_logger.info(f"Data Source: {DATA_SOURCE}")
audit_logger.info(f"Dataset Shape: {df.shape}")
audit_logger.info(f"Default Rate: {df['default'].mean():.2%}")

df.to_csv('data/fannie_mae_raw_sample.csv', index=False)
print("\n✅ Saved: data/fannie_mae_raw_sample.csv")
audit_logger.info("Saved: data/fannie_mae_raw_sample.csv")

print("\n" + "="*60)
print("SECTION 1 COMPLETE")
print("="*60)
audit_logger.info("SECTION 1 COMPLETE")


------------------------------------------------------------
EXECUTING DATA LOADING
------------------------------------------------------------
2026-09-07 10:51:22 | INFO | Executing data loading

Attempting to load data from: Fannie Mae.zip
2026-09-07 10:51:22 | INFO | Attempting to load data from: Fannie Mae.zip
  Found ZIP file: Fannie Mae.zip
2026-09-07 10:51:22 | INFO | Found ZIP file: Fannie Mae.zip
  Using CSV: train4.csv

  Reading data...
2026-09-07 10:51:22 | INFO | Reading CSV: train4.csv
  Total rows available: 1,672,822
2026-09-07 10:51:56 | INFO | Total rows available: 1,672,822
  Taking random sample of 100,000 rows...

  ✅ Data loaded successfully!
  Source: Real Fannie Mae Data (Random Sample)
  Shape: (100000, 133)
2026-09-07 10:51:57 | INFO | Data loaded: (100000, 133)
  Found target variable: 'TARGET' -> renamed to 'default'
2026-09-07 10:51:57 | INFO | Found target variable: 'TARGET' -> renamed to 'default'

  Creating PD-LGD estimates for real data...
2026-09-07

### SAVE RAW SAMPLE

In [12]:
df.to_csv('data/fannie_mae_raw_sample.csv', index=False)
print("\n✅ Saved: data/fannie_mae_raw_sample.csv")
audit_logger.info("Saved: data/fannie_mae_raw_sample.csv")

print("\n" + "="*60)
print("SECTION 1 COMPLETE")
print("="*60)
audit_logger.info("SECTION 1 COMPLETE")


✅ Saved: data/fannie_mae_raw_sample.csv
2026-09-07 10:52:11 | INFO | Saved: data/fannie_mae_raw_sample.csv

SECTION 1 COMPLETE
2026-09-07 10:52:11 | INFO | SECTION 1 COMPLETE


## SECTION 2: DATA QUALITY ASSESSMENT (with Safe Column Access)

#### 2.1 DATA OVERVIEW & STRUCTURE ANALYSIS

In [13]:
print("\n" + "-"*60)
print("SECTION 2: DATA OVERVIEW")
print("-"*60)

audit_logger.info("SECTION 2: DATA OVERVIEW")

print(f"\nDataset Shape: {df.shape}")
print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nData Types Distribution:")
dtype_summary = df.dtypes.value_counts()
for dtype, count in dtype_summary.items():
    print(f"  {dtype}: {count} columns")

# Identify column types
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
datetime_cols = df.select_dtypes(include=['datetime64']).columns.tolist()

print(f"\nColumn Categories:")
print(f"  Numeric Columns: {len(numeric_cols)}")
print(f"  Categorical Columns: {len(categorical_cols)}")
print(f"  Datetime Columns: {len(datetime_cols)}")

audit_logger.info(f"Numeric Columns: {len(numeric_cols)}")
audit_logger.info(f"Categorical Columns: {len(categorical_cols)}")
audit_logger.info(f"Datetime Columns: {len(datetime_cols)}")

# Check for enhancements
if 'risk_fico' in df.columns:
    print(f"\n✅ Non-linear risk factors present:")
    print(f"  - risk_fico: S-curve transformed FICO")
    print(f"  - risk_ltv: S-curve transformed LTV")
    if 'interaction_fico_ltv' in df.columns:
        print(f"  - interaction_fico_ltv: FICO-LTV interaction")
    audit_logger.info("Non-linear risk factors present")

if 'delinquency_state' in df.columns:
    print(f"\n✅ Multi-state delinquency indicators:")
    if 'delinquency_label' in df.columns:
        state_counts = df['delinquency_label'].value_counts()
        for state, count in state_counts.items():
            print(f"  - {state}: {count:,} ({count/len(df)*100:.1f}%)")
        audit_logger.info(f"Multi-state delinquency: {state_counts.to_dict()}")

if 'pd_estimate' in df.columns and 'lgd_final' in df.columns:
    pd_lgd_corr = df['pd_estimate'].corr(df['lgd_final'])
    print(f"\n✅ PD-LGD Correlation: {pd_lgd_corr:.3f}")
    if pd_lgd_corr > 0:
        print(f"  Positive correlation: Higher PD → Higher LGD")
    audit_logger.info(f"PD-LGD Correlation: {pd_lgd_corr:.3f}")



------------------------------------------------------------
SECTION 2: DATA OVERVIEW
------------------------------------------------------------
2026-09-07 10:52:20 | INFO | SECTION 2: DATA OVERVIEW

Dataset Shape: (100000, 142)
Memory Usage: 163.33 MB

Data Types Distribution:
  float64: 75 columns
  int64: 53 columns
  object: 13 columns
  int32: 1 columns

Column Categories:
  Numeric Columns: 129
  Categorical Columns: 13
  Datetime Columns: 0
2026-09-07 10:52:21 | INFO | Numeric Columns: 129
2026-09-07 10:52:21 | INFO | Categorical Columns: 13
2026-09-07 10:52:21 | INFO | Datetime Columns: 0

✅ Non-linear risk factors present:
  - risk_fico: S-curve transformed FICO
  - risk_ltv: S-curve transformed LTV
2026-09-07 10:52:21 | INFO | Non-linear risk factors present

✅ Multi-state delinquency indicators:
  - Current: 99,061 (99.1%)
  - 180+ DPD: 939 (0.9%)
2026-09-07 10:52:21 | INFO | Multi-state delinquency: {'Current': 99061, '180+ DPD': 939}

✅ PD-LGD Correlation: 0.119
  Posit

#### What This Means
- Columns: 136 - Original 133 + 3 added (delinquency indicators)
- Memory: 158.75 MB - Manageable for analysis
- Numeric: 123 columns - Mostly numerical data
- Categorical: 13 columns - Property type, occupancy, state, etc.
- Delinquency States: 2 - Current (99.1%), 180+ DPD (0.9%)

#### Key Decisions Explained
**1. Multi-State Delinquency Indicators**
I added multi-state delinquency indicators to capture more than just performing/defaulted. In the real data, we only had Current and 180+ DPD states because these are snapshot observations. The framework is designed to handle all 6 delinquency states when data is available.

**I implemented a multi-state delinquency framework that can handle 6 states (Current through 180+ DPD). In this dataset, we only observed Current and 180+ DPD, but the framework is ready for more granular delinquency data.**

#### Why This Matters:
- Binary default misses delinquency dynamics
- Multi-state enables more sophisticated modeling
- Framework is reusable for richer datasets

#### 2.2 TARGET VARIABLE VALIDATION

In [19]:
print("\n" + "-"*60)
print("SECTION 2.2: TARGET VARIABLE VALIDATION")
print("-"*60)

if 'default' in df.columns:
    print(f"\n✅ Target variable 'default' found")
    
    default_counts = df['default'].value_counts()
    default_rate = df['default'].mean()
    
    print(f"\nDefault Distribution:")
    print(f"  Good (0): {default_counts[0]:,} ({default_counts[0]/len(df):.1%})")
    print(f"  Bad  (1): {default_counts[1]:,} ({default_counts[1]/len(df):.1%})")
    print(f"  Default Rate: {default_rate:.2%}")
    
    if default_rate < 0.001:
        print("  ⚠️  WARNING: Default rate very low (< 0.1%)")
    elif default_rate > 0.10:
        print("  ⚠️  WARNING: Default rate very high (> 10%)")
    else:
        print("  ✅ Default rate within expected range (0.1% - 10%)")
    
    print(f"\nClass Imbalance Ratio: {default_counts[0]/default_counts[1]:.1f}:1")
    if default_counts[0]/default_counts[1] > 100:
        print("  ⚠️  High class imbalance may affect model performance")
    
    audit_logger.info(f"Default Rate: {default_rate:.2%}")
else:
    print("\n❌ ERROR: Target variable 'default' not found")
    audit_logger.error("Target variable 'default' not found")


------------------------------------------------------------
SECTION 2.2: TARGET VARIABLE VALIDATION
------------------------------------------------------------

✅ Target variable 'default' found

Default Distribution:
  Good (0): 99,061 (99.1%)
  Bad  (1): 939 (0.9%)
  Default Rate: 0.94%
  ✅ Default rate within expected range (0.1% - 10%)

Class Imbalance Ratio: 105.5:1
  ⚠️  High class imbalance may affect model performance
2026-09-07 11:21:29 | INFO | Default Rate: 0.94%


#### What It Means

This validates that the target variable (default) looks reasonable:

- 0.94% default rate (within the expected 0.1%-10% range)
- Class imbalance ratio: 105:1 (99.1% performing, 0.9% defaulted)

#### Why This Decision Was Made

- **Why validate the target variable?** - If the default rate is 0% or 100%, something is wrong with the data. This check catches data quality issues early.
- **Why the warning thresholds?** - 0.1% and 10% are industry sanity checks — mortgage default rates typically fall in this range.
- **Why the class imbalance warning?** - A 105:1 ratio means the model will be biased toward predicting "performing" because it sees far more performing loans. This affects model choice (you'd need to use techniques like SMOTE or class weights).

**I validate my data before modeling. I check for obvious issues and document them.**

### 2.3 MISSING VALUE ANALYSIS

In [20]:
print("\n" + "-"*60)
print("SECTION 2.3: MISSING VALUE ANALYSIS (OSFI E-23)")
print("-"*60)

missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': missing_counts.index,
    'Missing_Count': missing_counts.values,
    'Missing_Pct': missing_pct.values
})
missing_df = missing_df[missing_df['Missing_Count'] > 0]
missing_df = missing_df.sort_values('Missing_Count', ascending=False)

if len(missing_df) > 0:
    print(f"\nColumns with Missing Values: {len(missing_df)}")
    print("\nTop 10 Columns with Missing Values:")
    print(missing_df.head(10).to_string(index=False))
    
    high_missing = missing_df[missing_df['Missing_Pct'] > 10]
    medium_missing = missing_df[(missing_df['Missing_Pct'] >= 5) & (missing_df['Missing_Pct'] <= 10)]
    low_missing = missing_df[missing_df['Missing_Pct'] < 5]
    
    print(f"\nMissing Data Severity Categories:")
    print(f"  High (>10%): {len(high_missing)} columns")
    print(f"  Medium (5-10%): {len(medium_missing)} columns")
    print(f"  Low (<5%): {len(low_missing)} columns")
    
    """
    DECISION: Create missing indicators for all columns with missing values
    RATIONALE: OSFI E-23 requires tracking data quality issues
    REGULATORY REFERENCE: OSFI E-23 s.4.3
    """
    print("\nCreating Missing Indicators (OSFI E-23 s.4.3)...")
    missing_indicator_cols = []
    for col in missing_df['Column'].tolist():
        indicator_name = f'{col}_missing'
        df[indicator_name] = df[col].isnull().astype(int)
        missing_indicator_cols.append(indicator_name)
        print(f"  ✅ Created: {indicator_name}")
        audit_logger.info(f"Created missing indicator: {indicator_name}")
    
    print(f"\nCreated {len(missing_indicator_cols)} missing indicators")
    audit_logger.info(f"Created {len(missing_indicator_cols)} missing indicators")
else:
    print("\n✅ No missing values found")
    missing_df = pd.DataFrame(columns=['Column', 'Missing_Count', 'Missing_Pct'])
    audit_logger.info("No missing values found")


------------------------------------------------------------
SECTION 2.3: MISSING VALUE ANALYSIS (OSFI E-23)
------------------------------------------------------------

Columns with Missing Values: 36

Top 10 Columns with Missing Values:
               Column  Missing_Count  Missing_Pct
          FICO_CO_BOR            642        0.642
          pd_estimate            526        0.526
            lgd_final            526        0.526
       lgd_correlated            526        0.526
            risk_fico            521        0.521
             FICO_BOR            521        0.521
        FICO_ABS_DIFF            521        0.521
             FICO_MIN            429        0.429
 MATURITY_DIFF_STD_3M             16        0.016
MATURITY_DIFF_STD_12M             13        0.013

Missing Data Severity Categories:
  High (>10%): 0 columns
  Medium (5-10%): 0 columns
  Low (<5%): 36 columns

Creating Missing Indicators (OSFI E-23 s.4.3)...
  ✅ Created: FICO_CO_BOR_missing
2026-09-07 11:

#### What It Means

This identifies every column with missing values, shows how many values are missing, and creates a missing indicator for each column.

**Example:**

- FICO_CO_BOR: 642 missing (0.64%)
- pd_estimate: 526 missing (0.53%)

→ Creates FICO_CO_BOR_missing column (1 if missing, 0 if present)

#### Why This Decision Was Made
- **Why analyze missing values?** - OSFI E-23 s.4.3 requires data quality management. You can't manage what you don't measure.
- **Why create missing indicators?** - If you later find that pd_estimate_missing correlates with higher default rates, you know missing data is not random. This is an audit trail for data quality.
- **Why the severity categories (High/Medium/Low)?** - This helps prioritize — columns with >10% missing may need special handling (like dropping them).
- **Why display top 10?** - The top 10 likely capture all the important missing data. Showing all 37 columns would be overwhelming.

**I understand OSFI E-23 data quality requirements. I measure missing data, create audit trails, and prioritize issues by severity.**

### 2.4 DATA QUALITY REPORT

In [21]:
print("\n" + "-"*60)
print("SECTION 2.4: DATA QUALITY REPORT (OSFI E-23)")
print("-"*60)

quality_report_lines = []
quality_report_lines.append("="*80)
quality_report_lines.append("DATA QUALITY ASSESSMENT REPORT (ENHANCED)")
quality_report_lines.append("OSFI E-23 Compliance Documentation")
quality_report_lines.append("="*80)
quality_report_lines.append(f"\nReport Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
quality_report_lines.append(f"Run ID: {RUN_ID}")
quality_report_lines.append(f"Sample Size: {len(df):,} rows")

quality_report_lines.append("\n" + "-"*40)
quality_report_lines.append("MISSING DATA SUMMARY (OSFI E-23 s.4.3)")
quality_report_lines.append("-"*40)

if len(missing_df) > 0:
    quality_report_lines.append(f"\nTotal columns with missing values: {len(missing_df)}")
    quality_report_lines.append(f"Total missing values: {df.isnull().sum().sum():,}")
    quality_report_lines.append(f"Missing indicators created: {len(missing_indicator_cols)}")
else:
    quality_report_lines.append("\nNo missing values found")

quality_report_lines.append("\n" + "-"*40)
quality_report_lines.append("AUDIT TRAIL")
quality_report_lines.append("-"*40)
quality_report_lines.append(f"  Audit Log File: logs/ecl_audit_trail_{RUN_ID}.log")
quality_report_lines.append(f"  Run ID: {RUN_ID}")
quality_report_lines.append("  All assumptions logged with timestamps")

quality_report_text = "\n".join(quality_report_lines)
with open('outputs/data_quality_report_enhanced.txt', 'w', encoding='utf-8') as f:
    f.write(quality_report_text)
print("✅ Saved: outputs/data_quality_report_enhanced.txt")
audit_logger.info("Saved: outputs/data_quality_report_enhanced.txt")



------------------------------------------------------------
SECTION 2.4: DATA QUALITY REPORT (OSFI E-23)
------------------------------------------------------------
✅ Saved: outputs/data_quality_report_enhanced.txt
2026-09-07 11:21:36 | INFO | Saved: outputs/data_quality_report_enhanced.txt


### SECTION 3: DATA CLEANING (OSFI E-23 Compliant)

### 3.1 DOCUMENT CLEANING DECISIONS

In [22]:
print("\n" + "="*60)
print("SECTION 3: DATA CLEANING")
print("="*60)

audit_logger.info("SECTION 3: DATA CLEANING")


# ============================================================================
# 3.1 CLEANING DECISIONS DOCUMENTATION
# ============================================================================

print("\n" + "-"*60)
print("3.1 CLEANING DECISIONS DOCUMENTATION")
print("-"*60)

print("""
CLEANING METHODOLOGY (OSFI E-23 Compliant):

1. MISSING DATA THRESHOLDS
   ------------------------
   DECISION: Drop columns with >50% missing values
   RATIONALE: High missingness makes imputation unreliable
   REGULATORY REFERENCE: OSFI E-23 s.4.3
   RESULT: 0 columns dropped (all columns <50% missing)
   
   DECISION: Impute columns with <50% missing
   RATIONALE: Preserves data while addressing quality issues
   REGULATORY REFERENCE: OSFI E-23 s.4.3
   RESULT: 33 columns will be imputed

2. IMPUTATION METHOD FOR NUMERIC COLUMNS
   -------------------------------------
   DECISION: Use Median imputation
   RATIONALE: Robust to outliers, preserves distribution shape
   - Mean is sensitive to outliers
   - Median maintains central tendency
   - Mode is inappropriate for continuous variables
   REGULATORY REFERENCE: OSFI E-23 s.4.3

3. IMPUTATION METHOD FOR CATEGORICAL COLUMNS
   -----------------------------------------
   DECISION: Use Mode imputation
   RATIONALE: Most frequent value preserves category distribution
   - Maintains business meaning
   - Minimizes distortion of categorical relationships
   REGULATORY REFERENCE: OSFI E-23 s.4.3

4. IMPUTATION ORDER
   -----------------
   DECISION: Impute numeric columns first, then categorical
   RATIONALE: Prevents cascading effects
   - Numeric imputations don't depend on categorical
   - Some categorical imputations may use numeric features
   REGULATORY REFERENCE: OSFI E-23 s.4.3

5. OUTLIER TREATMENT
   ------------------
   DECISION: Winsorize at 1st and 99th percentiles
   RATIONALE: Preserves data while handling extreme values
   - Less destructive than removing outliers
   - Maintains sample size
   - Common in risk modeling
   REGULATORY REFERENCE: OSFI E-23 s.4.3
""")



SECTION 3: DATA CLEANING
2026-09-07 11:22:10 | INFO | SECTION 3: DATA CLEANING

------------------------------------------------------------
3.1 CLEANING DECISIONS DOCUMENTATION
------------------------------------------------------------

CLEANING METHODOLOGY (OSFI E-23 Compliant):

1. MISSING DATA THRESHOLDS
   ------------------------
   DECISION: Drop columns with >50% missing values
   RATIONALE: High missingness makes imputation unreliable
   REGULATORY REFERENCE: OSFI E-23 s.4.3
   RESULT: 0 columns dropped (all columns <50% missing)
   
   DECISION: Impute columns with <50% missing
   RATIONALE: Preserves data while addressing quality issues
   REGULATORY REFERENCE: OSFI E-23 s.4.3
   RESULT: 33 columns will be imputed

2. IMPUTATION METHOD FOR NUMERIC COLUMNS
   -------------------------------------
   DECISION: Use Median imputation
   RATIONALE: Robust to outliers, preserves distribution shape
   - Mean is sensitive to outliers
   - Median maintains central tendency
   - Mo

**Why This Matters:** This is your audit trail. Every decision is documented with rationale. An auditor can see exactly why you chose median vs mean, why you imputed numeric first, and why you winsorized at 1st/99th percentiles.

### 3.2 ANALYZE MISSING DATA BY CATEGORY

In [23]:
print("\n" + "-"*60)
print("3.2 MISSING DATA ANALYSIS BY CATEGORY")
print("-"*60)

# Separate missing columns by type
numeric_missing_cols = []
categorical_missing_cols = []

for col in missing_df['Column'].tolist():
    if col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_missing_cols.append(col)
        else:
            categorical_missing_cols.append(col)

print(f"\nColumns with Missing Values by Type:")
print(f"  Numeric Columns: {len(numeric_missing_cols)}")
print(f"  Categorical Columns: {len(categorical_missing_cols)}")

# Display sample of each
if numeric_missing_cols:
    print(f"\nSample Numeric Columns with Missing Values:")
    for col in numeric_missing_cols[:5]:
        missing_pct = (df[col].isnull().sum() / len(df)) * 100
        print(f"  {col}: {missing_pct:.2f}% missing")

if categorical_missing_cols:
    print(f"\nSample Categorical Columns with Missing Values:")
    for col in categorical_missing_cols[:5]:
        missing_pct = (df[col].isnull().sum() / len(df)) * 100
        print(f"  {col}: {missing_pct:.2f}% missing")


------------------------------------------------------------
3.2 MISSING DATA ANALYSIS BY CATEGORY
------------------------------------------------------------

Columns with Missing Values by Type:
  Numeric Columns: 36
  Categorical Columns: 0

Sample Numeric Columns with Missing Values:
  FICO_CO_BOR: 0.64% missing
  pd_estimate: 0.53% missing
  lgd_final: 0.53% missing
  lgd_correlated: 0.53% missing
  risk_fico: 0.52% missing


#### What It Means

This fills in missing values:

- **Numeric columns:** Replace missing values with the median (middle value)
- **Categorical columns:** Replace missing values with the mode (most frequent value)

#### Why This Decision Was Made

- **Why median for numeric?** - Median is robust to outliers. If you used mean, extreme values would skew the imputation. Median preserves the central tendency.
- **Why mode for categorical?** - Mode preserves the category distribution. If most loans are "SF" (Single Family), imputing "SF" is the most reasonable guess.
- **Why impute numeric first?** - Numeric imputation doesn't depend on categorical values. If you impute categorical first, it might affect numeric imputation — this order prevents cascading effects.
- **Why track before/after counts?** - This creates an audit trail — you can see exactly how many values were imputed for each column.

**I understand the tradeoffs of different imputation methods. I choose methods that are robust and defensible, and I document my choices.**

### 3.3 NUMERIC COLUMN IMPUTATION (Median)

In [24]:
print("\n" + "-"*60)
print("3.3 NUMERIC COLUMN IMPUTATION (Median)")
print("-"*60)

# Track imputation statistics
imputation_stats = {
    'numeric': [],
    'categorical': []
}

if numeric_missing_cols:
    print(f"\nImputing {len(numeric_missing_cols)} numeric columns...")
    
    for col in numeric_missing_cols:
        if col in df.columns:
            median_val = df[col].median()
            missing_before = df[col].isnull().sum()
            df[col].fillna(median_val, inplace=True)
            missing_after = df[col].isnull().sum()
            
            imputation_stats['numeric'].append({
                'column': col,
                'method': 'median',
                'value': median_val,
                'missing_before': missing_before,
                'missing_after': missing_after
            })
            
            print(f"  ✓ {col}: imputed {missing_before:,} missing values with median={median_val:.2f}")
    
    print(f"\n✅ Completed numeric imputation")
else:
    print("\n✅ No numeric columns with missing values to impute")


------------------------------------------------------------
3.3 NUMERIC COLUMN IMPUTATION (Median)
------------------------------------------------------------

Imputing 36 numeric columns...
  ✓ FICO_CO_BOR: imputed 642 missing values with median=786.00
  ✓ pd_estimate: imputed 526 missing values with median=0.06
  ✓ lgd_final: imputed 526 missing values with median=0.13
  ✓ lgd_correlated: imputed 526 missing values with median=0.15
  ✓ risk_fico: imputed 521 missing values with median=0.04
  ✓ FICO_BOR: imputed 521 missing values with median=786.00
  ✓ FICO_ABS_DIFF: imputed 521 missing values with median=0.00
  ✓ FICO_MIN: imputed 429 missing values with median=780.00
  ✓ MATURITY_DIFF_STD_3M: imputed 16 missing values with median=0.00
  ✓ MATURITY_DIFF_STD_12M: imputed 13 missing values with median=0.32
  ✓ MATURITY_DIFF_STD_6M: imputed 13 missing values with median=0.00
  ✓ UBP_CHANGE_STATUS_STD_12M: imputed 12 missing values with median=0.29
  ✓ DPD_DIFF_STD_3M: imputed 12 mis

#### What This Means for Missing vs. Imputed Value:
- **FICO_CO_BOR:** 642 / 786 - Median co-borrower score
- **FICO_BOR:** 521 / 786	- Median borrower score
- **FICO_ABS_DIFF:** 521 / 0 - Most borrowers have same score as co-borrower
- **FICO_MIN:** 429 / 780 - Median of minimum scores

**Key Insight:** The median FICO score is 786, which indicates a prime-quality portfolio. This is consistent with Fannie Mae's conventional loan standards.

### 3.4 CATEGORICAL COLUMN IMPUTATION (Mode)

In [25]:
print("\n" + "-"*60)
print("3.4 CATEGORICAL COLUMN IMPUTATION (Mode)")
print("-"*60)

if categorical_missing_cols:
    print(f"\nImputing {len(categorical_missing_cols)} categorical columns...")
    
    for col in categorical_missing_cols:
        if col in df.columns:
            mode_val = df[col].mode()[0] if not df[col].mode().empty else 'unknown'
            missing_before = df[col].isnull().sum()
            df[col].fillna(mode_val, inplace=True)
            missing_after = df[col].isnull().sum()
            
            imputation_stats['categorical'].append({
                'column': col,
                'method': 'mode',
                'value': mode_val,
                'missing_before': missing_before,
                'missing_after': missing_after
            })
            
            print(f"  ✓ {col}: imputed {missing_before:,} missing values with mode='{mode_val}'")
    
    print(f"\n✅ Completed categorical imputation")
else:
    print("\n✅ No categorical columns with missing values to impute")



------------------------------------------------------------
3.4 CATEGORICAL COLUMN IMPUTATION (Mode)
------------------------------------------------------------

✅ No categorical columns with missing values to impute


**What This Means:**
- SEL_NAME (Seller Name) had 1 missing value
- "OTHER" is the most frequent category
- This preserves the distribution

### 3.5 VALIDATE IMPUTATION RESULTS

In [26]:
print("\n" + "-"*60)
print("3.5 VALIDATE IMPUTATION RESULTS")
print("-"*60)

remaining_missing = df.isnull().sum().sum()
print(f"\nRemaining missing values after imputation: {remaining_missing}")

if remaining_missing == 0:
    print("✅ All missing values successfully imputed")
else:
    print("⚠️  Warning: Some missing values remain")
    remaining_cols = df.columns[df.isnull().any()].tolist()
    print(f"  Columns with remaining missing values: {remaining_cols}")

print(f"\nDataset shape after imputation: {df.shape}")
print(f"Columns added (missing indicators): {len([col for col in df.columns if '_missing' in col])}")


------------------------------------------------------------
3.5 VALIDATE IMPUTATION RESULTS
------------------------------------------------------------

Remaining missing values after imputation: 0
✅ All missing values successfully imputed

Dataset shape after imputation: (100000, 179)
Columns added (missing indicators): 37


### 3.6 OUTLIER TREATMENT (Winsorization)

In [27]:
print("\n" + "-"*60)
print("3.6 OUTLIER TREATMENT (Winsorization)")
print("-"*60)

def winsorize_column(data, col, lower_percentile=0.01, upper_percentile=0.99):
    """
    Winsorize a column to handle outliers.
    
    DECISION: Use 1st and 99th percentiles
    RATIONALE: 
    - 1st percentile catches extreme low values
    - 99th percentile catches extreme high values
    - This is a common risk management approach
    REGULATORY REFERENCE: OSFI E-23 s.4.3
    """
    if col not in data.columns:
        return data
    
    if not pd.api.types.is_numeric_dtype(data[col]):
        return data
    
    if data[col].nunique() <= 2:
        return data
    
    lower_bound = data[col].quantile(lower_percentile)
    upper_bound = data[col].quantile(upper_percentile)
    
    lower_outliers = (data[col] < lower_bound).sum()
    upper_outliers = (data[col] > upper_bound).sum()
    total_outliers = lower_outliers + upper_outliers
    
    data[col] = data[col].clip(lower=lower_bound, upper=upper_bound)
    
    outlier_info = {
        'column': col,
        'lower_bound': lower_bound,
        'upper_bound': upper_bound,
        'lower_outliers': lower_outliers,
        'upper_outliers': upper_outliers,
        'total_outliers': total_outliers,
        'pct_outliers': (total_outliers / len(data)) * 100
    }
    
    return data, outlier_info

key_columns_for_winsorization = ['ORG_UPB', 'ORG_LTV', 'DTI', 'FICO_BOR', 'AGE']
available_winsor_cols = [col for col in key_columns_for_winsorization if col in df.columns]

if available_winsor_cols:
    print(f"\nWinsorizing {len(available_winsor_cols)} key columns...")
    print("  (1st and 99th percentiles)")
    
    outlier_results = []
    for col in available_winsor_cols:
        df, outlier_info = winsorize_column(df, col)
        outlier_results.append(outlier_info)
        
        print(f"\n  {col}:")
        print(f"    Outliers treated: {outlier_info['total_outliers']:,} ({outlier_info['pct_outliers']:.2f}%)")
        print(f"    Bounds: [{outlier_info['lower_bound']:.2f}, {outlier_info['upper_bound']:.2f}]")
    
    print(f"\n✅ Outlier treatment complete")
    print(f"  Total outliers treated across all columns: {sum(r['total_outliers'] for r in outlier_results):,}")
else:
    print("\n✅ No key columns available for winsorization")



------------------------------------------------------------
3.6 OUTLIER TREATMENT (Winsorization)
------------------------------------------------------------

Winsorizing 5 key columns...
  (1st and 99th percentiles)

  ORG_UPB:
    Outliers treated: 1,811 (1.81%)
    Bounds: [40000.00, 625000.00]

  ORG_LTV:
    Outliers treated: 855 (0.85%)
    Bounds: [18.00, 97.00]

  DTI:
    Outliers treated: 838 (0.84%)
    Bounds: [9.00, 50.00]

  FICO_BOR:
    Outliers treated: 1,569 (1.57%)
    Bounds: [643.00, 820.00]

  AGE:
    Outliers treated: 856 (0.86%)
    Bounds: [7.00, 31.00]

✅ Outlier treatment complete
  Total outliers treated across all columns: 5,929


#### What It Means

This handles extreme values by capping them at the 1st and 99th percentiles.

Example for ORG_UPB (original loan amount):

- **1st percentile:** $40,000 (anything below becomes $40,000)
- **99th percentile:** $625,000 (anything above becomes $625,000)
- **1.81% of values were adjusted (1,811 loans)**

#### Why This Decision Was Made

- **Why winsorization instead of deletion?** - Deleting outliers removes data (reduces sample size). Winsorization keeps the data but limits extreme values.
- **Why 1st and 99th percentiles?** - This is a common risk management approach — it captures 98% of the data while handling the extremes.
- **Why not 5th and 95th?** - That would be too aggressive — you'd be changing too many values (10% of the data).
- **Why winsorize key columns only?** - These are the key risk drivers (loan amount, LTV, DTI, FICO, age). Other columns may not need it.

**I handle outliers in a way that preserves data while limiting extreme values. I document my approach and the impact.**

### 3.7 VERIFY CLEANING RESULTS

In [29]:
print("\n" + "-"*60)
print("3.7 VERIFY CLEANING RESULTS")
print("-"*60)

total_missing = df.isnull().sum().sum()
print(f"\nTotal Missing Values: {total_missing}")

if total_missing == 0:
    print("✅ All missing values handled")
else:
    print(f"⚠️  Remaining missing values: {total_missing}")
    for col in df.columns[df.isnull().any()]:
        print(f"    {col}: {df[col].isnull().sum():,} missing")

print("\nData Types After Cleaning:")
print(df.dtypes.value_counts())

duplicates = df.duplicated().sum()
print(f"\nDuplicate Rows: {duplicates:,}")


------------------------------------------------------------
3.7 VERIFY CLEANING RESULTS
------------------------------------------------------------

Total Missing Values: 0
✅ All missing values handled

Data Types After Cleaning:
float64    75
int64      53
int32      38
object     13
Name: count, dtype: int64

Duplicate Rows: 0


#### What This Means
- No missing values remain
- Data types are appropriate
- No duplicate rows found

#### What This Means - Mean vs. Median (Key Insights):
- **ORG_UPB:** $236K / $202K - Typical loan size is ~$200K
- **ORG_LTV:** 69% / 75% - Most loans have moderate leverage
- **DTI:** 30.8% / 31% - Borrowers have manageable debt
- **FICO_BOR:** 772 / 786 - Prime-quality portfolio
- **AGE:** 16 months / 15 months - Loans are relatively young

**Key Insight:** The portfolio is prime quality with average FICO of 772 and average LTV of 69%. This is a healthy portfolio.

#### Key Decisions Explained
**Decision 1: Median vs Mean Imputation**
I used median imputation because financial data often has extreme values. A mean would be pulled toward the outliers, while median maintains the central tendency of the distribution. This is standard practice in risk modeling.

**Decision 2: Numeric First, Then Categorical**
I imputed numeric columns first because they don't depend on categorical values. Then I imputed categorical columns, which might use numeric relationships in some cases. This prevents cascading effects.

**Decision 3: Winsorization at 1st/99th Percentiles**
I used winsorization rather than removing outliers because it maintains sample size while handling extreme values. The 1st and 99th percentiles are standard thresholds in risk modeling. This is less destructive than removing observations.

### 3.9 DOCUMENT CLEANING RESULTS (OSFI E-23)

In [30]:
print("\n" + "-"*60)
print("3.8 CLEANING DOCUMENTATION (OSFI E-23)")
print("-"*60)

cleaning_doc = []
cleaning_doc.append("="*80)
cleaning_doc.append("DATA CLEANING DOCUMENTATION")
cleaning_doc.append("OSFI E-23 Compliance")
cleaning_doc.append("="*80)
cleaning_doc.append(f"\nCleaning Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")

cleaning_doc.append("\n" + "-"*40)
cleaning_doc.append("1. CLEANING DECISIONS")
cleaning_doc.append("-"*40)
cleaning_doc.append("\nMissing Data Threshold: >50% missing -> Drop column")
cleaning_doc.append("  - Columns dropped: 0")
cleaning_doc.append("\nNumeric Imputation: Median")
cleaning_doc.append("  - Rationale: Robust to outliers, preserves distribution")
cleaning_doc.append("\nCategorical Imputation: Mode")
cleaning_doc.append("  - Rationale: Preserves category distribution")
cleaning_doc.append("\nImputation Order: Numeric first, then categorical")
cleaning_doc.append("  - Rationale: Prevents cascading effects")
cleaning_doc.append("\nOutlier Treatment: Winsorization (1st and 99th percentiles)")
cleaning_doc.append("  - Rationale: Less destructive than removal")

cleaning_doc.append("\n" + "-"*40)
cleaning_doc.append("2. NUMERIC IMPUTATION DETAILS")
cleaning_doc.append("-"*40)
if imputation_stats['numeric']:
    cleaning_doc.append("\nColumn                     Method    Imputed_Value  Missing_Before  Missing_After")
    for stat in imputation_stats['numeric']:
        cleaning_doc.append(f"{stat['column']:25} {stat['method']:8} {stat['value']:>12.2f}  {stat['missing_before']:>12,}   {stat['missing_after']:>12,}")
else:
    cleaning_doc.append("\nNo numeric imputation performed")

cleaning_doc.append("\n" + "-"*40)
cleaning_doc.append("3. CATEGORICAL IMPUTATION DETAILS")
cleaning_doc.append("-"*40)
if imputation_stats['categorical']:
    cleaning_doc.append("\nColumn                     Method    Imputed_Value  Missing_Before  Missing_After")
    for stat in imputation_stats['categorical']:
        cleaning_doc.append(f"{stat['column']:25} {stat['method']:8} {str(stat['value'])[:12]:>12}  {stat['missing_before']:>12,}   {stat['missing_after']:>12,}")
else:
    cleaning_doc.append("\nNo categorical imputation performed")

cleaning_doc.append("\n" + "-"*40)
cleaning_doc.append("4. OUTLIER TREATMENT DETAILS")
cleaning_doc.append("-"*40)
if outlier_results:
    cleaning_doc.append("\nColumn                  Lower_Bound  Upper_Bound  Outliers_Treated  Pct_Treated")
    for result in outlier_results:
        cleaning_doc.append(f"{result['column']:20} {result['lower_bound']:>10.2f}  {result['upper_bound']:>10.2f}  {result['total_outliers']:>10,}      {result['pct_outliers']:>6.2f}%")
else:
    cleaning_doc.append("\nNo outlier treatment performed")

cleaning_doc.append("\n" + "-"*40)
cleaning_doc.append("5. VALIDATION RESULTS")
cleaning_doc.append("-"*40)
cleaning_doc.append(f"\nFinal Shape: {df.shape}")
cleaning_doc.append(f"Total Missing Values: {total_missing}")
cleaning_doc.append(f"Duplicate Rows: {duplicates:,}")
cleaning_doc.append(f"Default Rate: {df['default'].mean():.2%}")
cleaning_doc.append("\n✅ All missing values successfully imputed")

cleaning_text = "\n".join(cleaning_doc)
with open('outputs/data_cleaning_documentation.txt', 'w', encoding='utf-8') as f:
    f.write(cleaning_text)
print("✅ Saved: outputs/data_cleaning_documentation.txt")

print("\n" + "="*60)
print("SECTION 3 COMPLETE - DATA CLEANING")
print("="*60)

audit_logger.info("SECTION 3 COMPLETE - DATA CLEANING")


------------------------------------------------------------
3.8 CLEANING DOCUMENTATION (OSFI E-23)
------------------------------------------------------------
✅ Saved: outputs/data_cleaning_documentation.txt

SECTION 3 COMPLETE - DATA CLEANING
2026-09-07 11:26:23 | INFO | SECTION 3 COMPLETE - DATA CLEANING


### SAVE FINAL DATA

In [31]:
print("\n" + "="*60)
print("SECTION 4: SAVE FINAL DATA")
print("="*60)

audit_logger.info("SECTION 4: SAVE FINAL DATA")

print("\nVerifying PD-LGD columns before saving:")
if 'pd_estimate' in df.columns:
    print(f"  ✅ pd_estimate: {df['pd_estimate'].mean():.4f}")
    audit_logger.info(f"pd_estimate: {df['pd_estimate'].mean():.4f}")
else:
    print("  ❌ pd_estimate: MISSING")
    audit_logger.warning("pd_estimate: MISSING")

if 'lgd_final' in df.columns:
    print(f"  ✅ lgd_final: {df['lgd_final'].mean():.4f}")
    audit_logger.info(f"lgd_final: {df['lgd_final'].mean():.4f}")
else:
    print("  ❌ lgd_final: MISSING")
    audit_logger.warning("lgd_final: MISSING")

if 'pd_estimate' in df.columns and 'lgd_final' in df.columns:
    pd_lgd_corr = df['pd_estimate'].corr(df['lgd_final'])
    print(f"  ✅ PD-LGD Correlation: {pd_lgd_corr:.3f}")
    audit_logger.info(f"PD-LGD Correlation: {pd_lgd_corr:.3f}")

# Save final data
df.to_csv('data/fannie_mae_final_clean.csv', index=False)
print("\n✅ Saved: data/fannie_mae_final_clean.csv")
print(f"   Shape: {df.shape}")
print(f"   Columns: {len(df.columns)}")
print(f"   PD-LGD columns: {'✅ Present' if ('pd_estimate' in df.columns and 'lgd_final' in df.columns) else '❌ Missing'}")
audit_logger.info(f"Saved: data/fannie_mae_final_clean.csv ({df.shape})")


SECTION 4: SAVE FINAL DATA
2026-09-07 11:26:28 | INFO | SECTION 4: SAVE FINAL DATA

Verifying PD-LGD columns before saving:
  ✅ pd_estimate: 0.0576
2026-09-07 11:26:28 | INFO | pd_estimate: 0.0576
  ✅ lgd_final: 0.1418
2026-09-07 11:26:28 | INFO | lgd_final: 0.1418
  ✅ PD-LGD Correlation: 0.119
2026-09-07 11:26:28 | INFO | PD-LGD Correlation: 0.119

✅ Saved: data/fannie_mae_final_clean.csv
   Shape: (100000, 179)
   Columns: 179
   PD-LGD columns: ✅ Present
2026-09-07 11:26:36 | INFO | Saved: data/fannie_mae_final_clean.csv ((100000, 179))


#### What It Means

This verifies that the PD-LGD columns were successfully created and then saves the final cleaned dataset for use in subsequent notebooks.

#### Why This Decision Was Made
- **Why verify PD-LGD columns?** - These are the most important columns for IFRS 9 modeling. If they're missing, the rest of the project fails. Verification catches errors early.
- **Why save as CSV?** - CSV is a universal format. The next notebooks (EDA, PD/LGD/EAD calibration) can load this data easily.
- **Why no index=True?** - You don't want to save the DataFrame index as a column — it would add an unnecessary Unnamed: 0 column.
- **Why this file name?** - fannie_mae_final_clean.csv clearly indicates this is the final cleaned version.
What a Hiring Manager Hears:

**I verify critical outputs before saving them. I use clear file names and save data in portable formats.**

### SECTION 11: NOTEBOOK LIMITATIONS

In [32]:
print("\n" + "="*60)
print("SECTION 5: NOTEBOOK LIMITATIONS")
print("="*60)

print("""
===============================================================================
NOTEBOOK 01 LIMITATIONS
===============================================================================

1. SYNTHETIC DATA FALLBACK
   -----------------------
   If the Fannie Mae ZIP file is not found, this notebook falls back to
   synthetic data. While this ensures the project runs, synthetic data
   does not capture real-world correlations or tail events.
   
   IMPACT: Results from synthetic data are for demonstration only and
   should not be interpreted as actual portfolio performance.
   
   MITIGATION: The project is designed to use real Fannie Mae data when
   available, and the same methodology applies to both data sources.

2. SAMPLE SIZE LIMITATION
   ----------------------
   This notebook uses a random sample of 100,000 rows. While this is
   sufficient for development and demonstration, a production environment
   would use the full dataset to capture rare events and tail risks.
   
   IMPACT: Rare default events may be underrepresented.
   
   MITIGATION: The random seed (42) ensures reproducibility, and the
   sample size is documented in the assumptions register.

3. DATA REPRESENTATIVENESS
   -----------------------
   This project uses Fannie Mae data as a PROXY for bank portfolio data.
   See the US/Canada Applicability Gap section for important differences.
   
   IMPACT: Results may not be representative of all portfolios.
   
   MITIGATION: The proxy nature of the data is explicitly documented
   and caveated throughout the project.

4. PD-LGD CORRELATION
   -------------------
   The PD-LGD correlation is created synthetically. In a production
   environment, this would be calibrated using historical recovery data.
   
   IMPACT: LGD estimates may not reflect actual recovery patterns.
   
   MITIGATION: The correlation structure is documented and can be
   recalibrated with portfolio-specific data.

5. OUTLIER TREATMENT
   ------------------
   Winsorization at the 1st and 99th percentiles is a common approach,
   but may not be appropriate for all variables or portfolios.
   
   IMPACT: Extreme values may be artificially compressed.
   
   MITIGATION: The approach is documented and can be adjusted with
   portfolio-specific validation.

6. MISSING DATA HANDLING
   ---------------------
   Median imputation for numeric columns and mode imputation for
   categorical columns are standard approaches, but may introduce bias.
   
   IMPACT: Imputed values may not reflect the true distribution.
   
   MITIGATION: Missing indicators are created for audit trail, and
   the imputation methodology is documented.

===============================================================================
""")


SECTION 5: NOTEBOOK LIMITATIONS

NOTEBOOK 01 LIMITATIONS

1. SYNTHETIC DATA FALLBACK
   -----------------------
   If the Fannie Mae ZIP file is not found, this notebook falls back to
   synthetic data. While this ensures the project runs, synthetic data
   does not capture real-world correlations or tail events.
   
   IMPACT: Results from synthetic data are for demonstration only and
   should not be interpreted as actual portfolio performance.
   
   MITIGATION: The project is designed to use real Fannie Mae data when
   available, and the same methodology applies to both data sources.

2. SAMPLE SIZE LIMITATION
   ----------------------
   This notebook uses a random sample of 100,000 rows. While this is
   sufficient for development and demonstration, a production environment
   would use the full dataset to capture rare events and tail risks.
   
   IMPACT: Rare default events may be underrepresented.
   
   MITIGATION: The random seed (42) ensures reproducibility, and the
   sa

#### What It Means

This is a self-assessment of the limitations of this notebook. It shows you understand where your analysis might fall short.

#### Why This Decision Was Made:
- **Why list limitations?** - OSFI E-23 requires intellectual honesty. If you don't acknowledge limitations, a regulator will find them anyway. It's better to be transparent.
- **Why include impact?** - "Impact" tells the reader why this limitation matters — not just that it exists, but what it means for the results.
- **Why include mitigation?** - "Mitigation" shows you're proactive — you're not just identifying problems; you're addressing them.

**I am intellectually honest and self-aware. I don't claim perfection — I acknowledge limitations and explain how I mitigate them.**

### SECTION 12: NOTEBOOK COMPLETE

In [33]:
print("\n" + "="*80)
print("NOTEBOOK 01 COMPLETE - DATA PREPARATION PIPELINE")
print("="*80)
print("""
Data Preparation Pipeline Summary:
  ✅ Data Loading: Random sample of 100,000 rows
  ✅ Data Quality: 33 missing indicators created (OSFI E-23 s.4.3)
  ✅ Data Cleaning: Numeric imputations (Median), Categorical imputations (Mode)
  ✅ Outlier Treatment: Winsorization at 1st/99th percentiles
  ✅ PD-LGD Estimates: Created and validated
  ✅ Assumptions Register: Created and saved
  ✅ CECL vs IFRS 9: Context documented
  ✅ US/Canada Gap: Applicability differences documented
  ✅ Limitations: Documented

Files Generated:
  - data/fannie_mae_raw_sample.csv
  - data/fannie_mae_cleaned.csv
  - data/fannie_mae_final_clean.csv
  - data/assumptions_register.csv
  - outputs/data_quality_report_enhanced.txt
  - outputs/data_cleaning_documentation.txt
  - logs/ecl_audit_trail_{RUN_ID}.log

Next Step: Open NOTEBOOK 02 - EDA and Risk Factor Analysis
""")
print("="*80)

audit_logger.info("NOTEBOOK 01 COMPLETE")


NOTEBOOK 01 COMPLETE - DATA PREPARATION PIPELINE

Data Preparation Pipeline Summary:
  ✅ Data Loading: Random sample of 100,000 rows
  ✅ Data Quality: 33 missing indicators created (OSFI E-23 s.4.3)
  ✅ Data Cleaning: Numeric imputations (Median), Categorical imputations (Mode)
  ✅ Outlier Treatment: Winsorization at 1st/99th percentiles
  ✅ PD-LGD Estimates: Created and validated
  ✅ Assumptions Register: Created and saved
  ✅ CECL vs IFRS 9: Context documented
  ✅ US/Canada Gap: Applicability differences documented
  ✅ Limitations: Documented

Files Generated:
  - data/fannie_mae_raw_sample.csv
  - data/fannie_mae_cleaned.csv
  - data/fannie_mae_final_clean.csv
  - data/assumptions_register.csv
  - outputs/data_quality_report_enhanced.txt
  - outputs/data_cleaning_documentation.txt
  - logs/ecl_audit_trail_{RUN_ID}.log

Next Step: Open NOTEBOOK 02 - EDA and Risk Factor Analysis

2026-09-07 11:29:10 | INFO | NOTEBOOK 01 COMPLETE


**I document my work clearly. I provide summaries that anyone can understand.**